In [0]:
%pip install kagglehub

In [0]:
import os
import tempfile
import kagglehub
from kagglehub import KaggleDatasetAdapter

### How is it done?
I got one huge file from kaggle and saved it into my Volume folder. Then I limited my dataframe to 1000 rows (because I will create 1000 one row csv files). Then I iterated through my dataframe and created these files. I used ",".join instead of creating 1000 seperate dataframes because I assume that it`s more efficient. I did all of this this way because I do not have access to read file from .cache/kagglehub or temporary folders. Finally I deleted one huge file, because it necesarilly takes place in my storage.
 <br>PS First files are the newest ones so I will start from reading file_1000 then file_999 .... to reach the newest ones in the end

In [0]:
SECRET_SCOPE = "kv-blech-dev" 
API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key="kaggle-token")

os.environ['KAGGLE_USERNAME'] = 'lechster'
os.environ['KAGGLE_KEY'] = API_KEY

In [0]:
# path for downloading file
# Im using this folder because I`ve got no access to read file from .cache/kagglehub or temporary folders

volume_path = "/Volumes/dbr_dev/lechster10_bronze/lab3_volume"
my_tmp_dir = f"{volume_path}/streaming_data/tmp"

path_to_data = kagglehub.dataset_download(
        'prasoonkottarathil/btcinusd', 
        output_dir=my_tmp_dir
    )    

my_file = "BTC-Daily.csv"
df = spark.read.format("csv").option("header", True).load(f"{path_to_data}/{my_file}")


In [0]:
df = df.limit(1000)
display(df)

In [0]:
for col in df.columns:
    if " " in col:
        old_col = col
        col = col.replace(" ", "_")
        df = df.withColumnRenamed(old_col, col)



In [0]:
df.columns

In [0]:
rows = df.collect()
rows = rows[::-1]  # I want the oldest ones first

col_line = ",".join([str(col) if col is not None else "" for col in df.columns])   

for i, row in enumerate(rows):        
    csv_lines = col_line + "\n" + ",".join([str(val) if val is not None else "" for val in row])       
    dbutils.fs.put(f"{volume_path}/streaming_data/file_{i+1:04d}.csv", csv_lines, overwrite=True)
